In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from loguru import logger

ROOT = Path.cwd().resolve()

while ROOT != ROOT.parent and not (ROOT / "app").is_dir():
    ROOT = ROOT.parent

if not (ROOT / "app").is_dir():
    raise RuntimeError(
        "Папка app не найдена. "
        "Откройте Notebook внутри репозитория CloudMarks."
    )

sys.path.insert(0, str(ROOT))

In [2]:
from app.batch import (
    ExtractionConfig,
    PointPairComparisonRunner,
)

logger.remove()

config = ExtractionConfig(
    default_radius=0.60,
    min_neighborhood_points=600,
    min_points_per_plane=200,
    max_reference_distance_factor=1.25,
    normal_k=8,
    cluster_eps=0.08,
    cluster_min_samples=3,
)

runner = PointPairComparisonRunner.from_files(
    epoch1_path="../data/t2/scan_2335_split_a.las",
    epoch2_path="../data/t2/scan_2335_split_b.las",
    config=config,
    alpha=0.05,
    max_pair_distance=0.03,
)

results = runner.run_from_reference_file(
    reference_points_path="../data/t2/vse_tochki.txt",
    show_progress=True,
)

results

Обработка точек: 100%|██████████| 392/392 [00:52<00:00,  7.47эпоха/s, OK=32]

Анализ деформаций...
Готово: 32/196 пар


,name,reference_x,reference_y,reference_z,radius,epoch1_neighborhood_points,epoch2_neighborhood_points,epoch1_reference_distance,epoch2_reference_distance,pair_distance,...,sigma_dz,sigma_displacement,sigma_displacement_mm,t_value,p_value_t,significant_t,chi2_value,p_value_chi2,significant_chi2,analysis_reliable
0,A_10_2_l,99.3127,107.7533,8.1465,0.6,12029,11984,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A_10_2_r,101.0595,107.7900,8.1438,0.6,11997,11971,0.024443,0.024523,0.000378,...,0.000075,0.00021,0.209549,1.80592,0.070931,False,4.77058,0.189388,False,True
2,A_10_3_l,99.3061,107.7612,11.6365,0.6,7988,8017,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A_10_3_r,101.0720,107.7561,11.6403,0.6,8119,8061,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A_10_4_l,99.2973,107.7508,15.3065,0.6,5398,5405,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,A_9_6_r,98.0572,107.7900,22.4644,0.6,2927,2897,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
192,A_9_7_l,96.2911,107.7365,25.9726,0.6,2227,2225,1.060185,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193,A_9_7_r,98.0578,107.7522,25.9854,0.6,2288,2278,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194,A_9_8_l,96.2968,107.7472,29.5982,0.6,1858,1882,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
deformations = results.query(
    "processing_status == 'SUCCESS'"
)

deformations.head(10)

,name,reference_x,reference_y,reference_z,radius,epoch1_neighborhood_points,epoch2_neighborhood_points,epoch1_reference_distance,epoch2_reference_distance,pair_distance,...,sigma_dz,sigma_displacement,sigma_displacement_mm,t_value,p_value_t,significant_t,chi2_value,p_value_chi2,significant_chi2,analysis_reliable
1,A_10_2_r,101.0595,107.7900,8.1438,0.6,11997,11971,0.024443,0.024523,0.000378,...,0.000075,0.000210,0.209549,1.805920,0.070931,False,4.770580,1.893881e-01,False,True
17,A_11_3_r,104.0912,107.7559,11.6290,0.6,6943,6889,0.017506,0.017820,0.000376,...,0.000100,0.000100,0.100451,3.741223,0.000183,True,15.080056,1.749498e-03,True,True
33,A_12_4_r,107.0961,107.7900,15.3260,0.6,3751,3745,0.030963,0.031013,0.000329,...,0.000136,0.000171,0.170959,1.926228,0.054076,False,4.222262,2.384425e-01,False,True
35,A_12_5_r,107.1023,107.7518,18.8447,0.6,2993,2977,0.016223,0.015476,0.000916,...,0.000189,0.000197,0.196954,4.650041,0.000003,True,23.234863,3.607739e-05,True,True
37,A_12_6_r,107.1049,107.7444,22.4630,0.6,2298,2320,0.017730,0.018246,0.000596,...,0.000205,0.000205,0.204883,2.908606,0.003630,True,12.865695,4.936235e-03,True,True
43,A_13_2_r,110.0912,107.7668,8.1473,0.6,4350,4340,0.008577,0.008849,0.000297,...,0.000130,0.000125,0.125131,2.377104,0.017449,True,28.592877,2.726883e-06,True,True
45,A_13_3_r,110.0922,107.7900,11.6324,0.6,3417,3393,0.033980,0.033673,0.000373,...,0.000195,0.000148,0.148054,2.517474,0.011820,True,33.712789,2.277985e-07,True,True
47,A_13_4_r,110.0968,107.7900,15.3104,0.6,2829,2817,0.035062,0.034929,0.000226,...,0.000194,0.000172,0.171804,1.314223,0.188771,False,5.505586,1.383049e-01,False,True
49,A_13_5_r,110.0993,107.7900,18.8398,0.6,2278,2297,0.035137,0.035434,0.000792,...,0.000212,0.000209,0.209426,3.780506,0.000157,True,26.591654,7.169772e-06,True,True
51,A_13_6_r,110.0975,107.7459,22.4508,0.6,1833,1834,0.010921,0.010481,0.001080,...,0.000234,0.000367,0.367291,2.940391,0.003278,True,10.155329,1.729090e-02,True,True


In [4]:
deformations
deformations = (
    results.query("processing_status == 'SUCCESS'")
    .sort_values(
        by="displacement_mm",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

deformations[
    [
        "name",
        "epoch1_x",
        "epoch1_y",
        "epoch1_z",
        "epoch2_x",
        "epoch2_y",
        "epoch2_z",
        "dx",
        "dy",
        "dz",
        "displacement_mm",
    ]
].round(4)

,name,epoch1_x,epoch1_y,epoch1_z,epoch2_x,epoch2_y,epoch2_z,dx,dy,dz,displacement_mm
0,A_15_6_r,116.1194,107.7623,22.4624,116.1198,107.7622,22.4637,0.0005,-0.0001,0.0013,1.4052
1,A_16_5_r,119.1194,107.7637,18.8428,119.1180,107.7638,18.8427,-0.0014,0.0001,-0.0001,1.3834
2,A_6_6_l,87.2826,107.7562,22.4527,87.2839,107.7564,22.4531,0.0013,0.0002,0.0004,1.3358
3,A_14_4_r,113.1114,107.7575,15.3233,113.1125,107.7575,15.3226,0.0011,0.0001,-0.0006,1.2711
4,A_5_4_l,84.2751,107.7660,15.3265,84.2762,107.7660,15.3261,0.0011,-0.0000,-0.0004,1.1971
5,A_13_6_r,110.0936,107.7515,22.4593,110.0947,107.7514,22.4593,0.0011,-0.0001,-0.0001,1.0800
6,A_8_3_l,93.2875,107.7624,11.6431,93.2877,107.7626,11.6422,0.0003,0.0001,-0.0009,0.9509
7,A_6_2_l,87.2774,107.7652,8.1463,87.2773,107.7652,8.1473,-0.0001,0.0000,0.0009,0.9242
8,A_12_5_r,107.1047,107.7580,18.8595,107.1042,107.7581,18.8587,-0.0004,0.0001,-0.0008,0.9158
9,A_15_5_r,116.1250,107.7645,18.8595,116.1243,107.7644,18.8593,-0.0008,-0.0000,-0.0002,0.7945


In [5]:
from app.batch import (
    ExtractionConfig,
    SingleScanPointExtractor,
)

config = ExtractionConfig(
    default_radius=0.60,
)

extractor = SingleScanPointExtractor.from_files(
    scan_path="../data/t2/scan_2335.las",
    reference_points_path="../data/t2/vse_tochki.txt",
    config=config,
)

extractor.run()
extractor.export_csv("output/virtual_points_epoch1.csv")

[1/3] Загрузка скана... 

Загрузка scan_2335.las: 100%|██████████| 4337974/4337974 [00:11<00:00, 390740.28точка/s]

готово: 4,337,974 точек
[2/3] Построение пространственного индекса... 

готово
Опорных точек: 196


[3/3] Извлечение точек: 100%|██████████| 196/196 [01:21<00:00,  2.39точка/s]

Готово: 96/196 надёжных точек


PosixPath('output/virtual_points_epoch1.csv')

In [6]:
results = extractor.to_dataframe()

results

,name,reference_x,reference_y,reference_z,radius,neighborhood_points,reference_distance,status,message,x,y,z,geometry_status,reliable_accuracy,sigma_x,sigma_y,sigma_z
0,A_10_2_l,99.3127,107.7533,8.1465,0.6,24013,0.015272,SUCCESS,OK,99.313273,107.766833,8.153555,GOOD,True,0.000120,0.000007,0.000037
1,A_10_2_r,101.0595,107.7900,8.1438,0.6,23968,NaN,UNRELIABLE,Точка ненадёжна: geometry_status=PARALLEL.,102.509890,107.767083,10.094516,PARALLEL,False,NaN,NaN,NaN
2,A_10_3_l,99.3061,107.7612,11.6365,0.6,16005,NaN,UNRELIABLE,Точка ненадёжна: geometry_status=PARALLEL.,104.881809,107.763380,11.659149,PARALLEL,False,NaN,NaN,NaN
3,A_10_3_r,101.0720,107.7561,11.6403,0.6,16180,0.013129,SUCCESS,OK,101.069974,107.766296,11.648319,GOOD,True,0.000079,0.000013,0.000046
4,A_10_4_l,99.2973,107.7508,15.3065,0.6,10803,0.013307,SUCCESS,OK,99.297702,107.760304,15.315805,GOOD,True,0.000150,0.000013,0.000055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,A_9_6_r,98.0572,107.7900,22.4644,0.6,5824,NaN,UNRELIABLE,Точка ненадёжна: geometry_status=PARALLEL.,96.511146,107.755435,22.477595,PARALLEL,False,NaN,NaN,NaN
192,A_9_7_l,96.2911,107.7365,25.9726,0.6,4452,0.018899,SUCCESS,OK,96.285084,107.745546,25.988064,GOOD,True,0.000132,0.000044,0.000100
193,A_9_7_r,98.0578,107.7522,25.9854,0.6,4566,NaN,UNRELIABLE,Точка ненадёжна: geometry_status=PARALLEL.,98.705534,107.745885,25.991013,PARALLEL,False,NaN,NaN,NaN
194,A_9_8_l,96.2968,107.7472,29.5982,0.6,3740,0.007856,SUCCESS,OK,96.292480,107.746741,29.604745,GOOD,True,0.000282,0.000038,0.000139


In [7]:
from app.cross_points.CrossPointExacter import CrossPointExacter


exacter = CrossPointExacter(
    file_path="../data/t1/1_A_10_2_l.txt",
    show_scans=False,
    normal_k=8,
    min_points_per_plane=100,
    cluster_eps=0.08,
)

point = exacter.calculate_intersect_point()
point.name = "A_10_2_l"

df_point = point.to_dataframe()
display(df_point)

,name,x,y,z,status,reliable_accuracy,mse,plane_1_mse,plane_2_mse,plane_3_mse,sigma_x,sigma_y,sigma_z,ellipsoid_confidence,ellipsoid_a,ellipsoid_b,ellipsoid_c
0,A_10_2_l,99.313348,107.766052,8.153626,GOOD,True,0.000189,0.000529,0.000794,0.000557,0.000179,0.000015,0.000057,0.95,0.000501,0.00016,0.000042
